# GeoProspectNet — End-to-End Pipeline

**Continental-scale geothermal discovery via multi-modal contrastive deep learning across the western United States.**

This notebook reproduces every table and figure in the manuscript:

1. Load curated continental dataset (235 K × 4 km cells, six modalities)
2. Train the four model variants (cpu_calibrated, cpu_tuned, cpu_margin, cpu_max)
3. Validate against five cohorts (positives, post-2008 hold-out, NC1/NC3, random)
4. Run discovery with the two best variants and take the consensus set
5. Quantify recoverable resource (USGS Williams 2008 volumetric method)
6. Benchmark against classical baselines (XGBoost, Random Forest, Logistic Regression, heat-flow rule)
7. Permutation-importance modality ablation
8. Calibration analysis (Brier, ECE, reliability diagram)
9. Generate publication figures

**Reproducibility:** All seeds are pinned (default seed 42). The full pipeline runs in ~3 hours on CPU; pre-computed checkpoints and continental scores live under `outputs/` and short-circuit any step you don't want to re-run. Data sources are real and public — see the manuscript Data Availability statement.

**Hardware:** developed on a MacBook (no GPU). Each training run is 5–30 minutes. The continental inference passes (~235 K cells) are the longest single steps.

## 0  Setup

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
import numpy as np, pandas as pd, yaml, torch

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams.update({'figure.dpi': 100, 'savefig.dpi': 300,
                     'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False})

print(f'project root: {ROOT}')
print(f'device: {"cuda" if torch.cuda.is_available() else "cpu"}')
print(f'torch {torch.__version__}, numpy {np.__version__}')

## 1  Data inventory

Six public datasets are pulled in `src/data/`:

| modality | source | shape per cell |
|---|---|---|
| Geophysics | USGS GMNA gravity/magnetic, SMU heat flow, SRTM elevation | 6 channels × 32 × 32 patch |
| Geochemistry | USGS GEOTHERM springs | 9 features |
| Geology | USGS SGMC lithology, Mordensky 2023 faults | 11 features |
| Labels | USGS Williams 2008 (positives) + heuristic negatives | int8 |

Grid: 4 km lat/lon over 31–49°N × −125 to −103°W = **235 470 cells**.

In [ ]:
processed = ROOT / 'data/processed'
grid = pd.read_csv(processed / 'grid_coordinates.csv')
labels = np.load(processed / 'labels.npy')
train_mask = np.load(processed / 'train_mask.npy')

print(f'Grid cells:     {len(grid):,}')
print(f'Positive cells: {int((labels == 1).sum()):,}')
print(f'Labeled neg:    {int(train_mask.sum() - (labels == 1).sum()):,}')
print(f'Unlabeled:      {int((~train_mask).sum()):,}')
print(f'Total cells:    {len(grid):,}')

## 2  Training (skip if `outputs/checkpoints/random_cpu_*_seed42.pt` exists)

In [ ]:
CONFIGS = ['cpu_calibrated', 'cpu_tuned', 'cpu_margin', 'cpu_max']
for name in CONFIGS:
    ckpt = ROOT / f'outputs/checkpoints/random_{name}_seed42.pt'
    if ckpt.exists():
        print(f'  ✓ {name:18s} cached  ({ckpt.stat().st_size // 1024} KB)')
    else:
        print(f'  ✗ {name:18s} missing — uncomment cell below to retrain')

# Uncomment to retrain everything from scratch (each run is 5–30 min on CPU):
# for name in CONFIGS:
#     subprocess.check_call(['python', '-m', 'src.training.train',
#                            '--config', f'configs/{name}.yaml',
#                            '--random', '--seed', '42'])

## 3  Continental inference (cpu_max)

Scores every cell in the western US grid. Cached at `outputs/results/scores_cpu_max.npy`.

In [ ]:
scores_path = ROOT / 'outputs/results/scores_cpu_max.npy'
if scores_path.exists():
    scores = np.load(scores_path)
    print(f'loaded cached scores: n={len(scores):,} median={np.median(scores):.4f} mean={scores.mean():.4f}')
else:
    print('Run: python -m src.evaluation.negative_controls --config configs/cpu_max.yaml --ckpt outputs/checkpoints/random_cpu_max_seed42.pt')

## 4  Cohort validation

Score five cohorts on the percentile scale of the continental grid:
1. Known USGS positives (1 370 cells)
2. Post-2008 blind hold-out (22 cells — including Zanskar Big Blind)
3. Random western US (100 cells)
4. NC1 Colorado Plateau interior (known-cold)
5. NC3 random deep (>100 km from any field)

A healthy model puts (1) and (2) near the top, (3) near the median, and (4–5) near the bottom.

In [ ]:
nc = pd.read_csv(ROOT / 'outputs/results/negative_controls_cpu_max.csv')
print(nc[['cohort','n','mean_p','mean_percentile','frac_above_90pct']].to_string(index=False))

print()
print(f'Separation gap (positives mean pct − NC3 mean pct):',
      f"{nc.loc[nc.cohort=='known_positives','mean_percentile'].iat[0] - nc.loc[nc.cohort=='NC3_random_deep','mean_percentile'].iat[0]:.1f}")

## 5  Baseline comparison

Classical ML methods on the same labeled data. AUROC is perfect for every method, but the **post-2008 hold-out** exposes overfitting.

In [ ]:
bl = pd.read_csv(ROOT / 'outputs/results/baselines.csv')
bl = bl[['method','auroc','auprc','capture_top5pct','holdout_mean_pct','holdout_top10_capture']]
print(bl.to_string(index=False, float_format='%.3f'))

## 6  Loss / data / architecture ablations

Three independent sweeps:
1. **Loss**: BCE (cpu_calibrated) → focal (cpu_tuned) → margin (cpu_margin)
2. **Negative-pool size**: 5× (calibrated) → 10× (tuned) → ~20× (max)
3. **Architecture**: contrastive on/off, spatial smoothing on/off (`src/evaluation/ablation.py`)

Headline metric: percentile separation between known positives and random-deep negatives.

In [ ]:
sweep = pd.read_csv(ROOT / 'outputs/results/separation_sweep.csv')
print('--- loss + data ablation (cpu_calibrated → cpu_tuned → cpu_margin) ---')
print(sweep[['config','pos_mean_pct','nc3_mean_pct','separation_gap','holdout_mean_pct']]
      .to_string(index=False, float_format='%.2f'))

## 7  Modality permutation importance

Shuffle each modality's features across all cells, re-score, measure ΔAUROC. Mean ± std over 5 repeats.

In [ ]:
perm_path = ROOT / 'outputs/results/permutation_importance.csv'
if perm_path.exists():
    perm = pd.read_csv(perm_path)
    print(perm.to_string(index=False, float_format='%.4f'))
else:
    print('Run: python -m src.evaluation.modality_analysis --config configs/cpu_max.yaml --ckpt outputs/checkpoints/random_cpu_max_seed42.pt --n_repeats 5')

## 8  Calibration analysis

In [ ]:
cal = pd.read_csv(ROOT / 'outputs/results/calibration_table.csv')
summary = json.load(open(ROOT / 'outputs/results/calibration_summary.json'))
print('Brier score: %.4f' % summary['brier_score'])
print('ECE:         %.4f' % summary['expected_calibration_error'])
print()
print(cal.to_string(index=False, float_format='%.4f'))

## 9  Discovery — 33 consensus candidate sites with MWe estimates

In [ ]:
mwe = pd.read_csv(ROOT / 'outputs/results/consensus_with_mwe.csv')
mwe = mwe.sort_values('mwe_central', ascending=False)
print('=== TOP 10 NEW DISCOVERY SITES BY MWe (consensus, cpu_margin ∩ cpu_max) ===')
print(mwe.head(10)[['discovery_id','lat','lon','area_km2','T_res_estimate_C',
                    'mwe_central','province']].to_string(index=False, float_format='%.1f'))
print()
print(f'Cumulative recoverable resource:')
print(f'  central   {mwe.mwe_central.sum():,.0f} MWe')
print(f'  low       {mwe.mwe_low.sum():,.0f} MWe')
print(f'  high      {mwe.mwe_high.sum():,.0f} MWe')
print()
print(f'Comparison: US installed geothermal capacity ≈ 3 800 MWe (2023).')

## 10  Publication figures

Eight figures generated to `outputs/figures/`:

1. Continental prospectivity map
2. Uncertainty map (MC-Dropout std)
3. Cohort score distributions
4. ROC + PR curves
5. Calibration / reliability diagram
6. Modality permutation importance bar chart
7. MWe per discovery (histogram + cumulative)
8. Discovery cluster gallery (4-6 top sites with geological context)

In [ ]:
subprocess.check_call(['python', '-m', 'src.visualization.make_figures'])

## 11  Summary numbers for the paper

In [ ]:
print('==== ABSTRACT NUMBERS ====')
nc = pd.read_csv(ROOT / 'outputs/results/negative_controls_cpu_max.csv')
mwe = pd.read_csv(ROOT / 'outputs/results/consensus_with_mwe.csv')
bl = pd.read_csv(ROOT / 'outputs/results/baselines.csv')
ho = float(nc.loc[nc.cohort == 'temporal_holdout_post2008', 'mean_percentile'].iat[0])
gap = float(nc.loc[nc.cohort == 'known_positives', 'mean_percentile'].iat[0]
            - nc.loc[nc.cohort == 'NC3_random_deep', 'mean_percentile'].iat[0])
print(f'  • Cells scored:                {235_470:,}')
print(f'  • Known fields (training):     280 (1 370 cells)')
print(f'  • Post-2008 hold-out sites:    22 (all in top 10%)')
print(f'  • Hold-out mean percentile:    {ho:.1f}')
print(f'  • Positive-NC3 separation:     {gap:.1f} percentile points')
print(f'  • Consensus discoveries:       {len(mwe)}')
print(f'  • Cumulative MWe (central):    {mwe.mwe_central.sum():,.0f}')
print(f'  • MWe range:                   [{mwe.mwe_low.sum():,.0f} – {mwe.mwe_high.sum():,.0f}]')
print(f'  • US installed (2023):         ≈ 3 800')
print(f'  • Implied capacity multiplier: {mwe.mwe_central.sum() / 3800:.1f}×')
print(f'  • Best baseline hold-out:      {float(bl.holdout_mean_pct.iloc[1]):.1f} ({bl.method.iloc[1]})')
print(f'  • Tree-based ML hold-out:      ~60 (capture@10% = 0%) — overfit')

---
## Citation

If you use this pipeline, please cite the manuscript and the data sources listed in `DATA.md`. The code is released under the MIT license at `LICENSE`.